# E04: 
we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

# E05
look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

In [1]:
import torch
import torch.nn.functional as F

In [2]:
words = open("names.txt", 'r').read().splitlines()
words[:5], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia'], 32033)

In [3]:
min(words, key=len), len(min(words, key=len)), max(words, key=len), len(max(words, key=len))

('an', 2, 'muhammadibrahim', 15)

In [4]:
# Get list of all the characters
chars = ['.'] + sorted(list(set(''.join(words)))) # ., a, b, c, d, .... x, y, z

# Create dictionary for mapping single character to its ID
stoi = {s:i for i, s in enumerate(chars)}

# Create another dictionary for reverse mapping
itos = {i:s for s, i in stoi.items()}

# Check if its correct
len(chars), stoi['.'], stoi['a'], itos[0]

(27, 0, 1, '.')

In [5]:
# Get all 27*27 combinations (26 alphabets and one '.')
all_combinations = [a + b for a in chars for b in chars]

# Create dictionary for mapping combination to its ID
ctoi={c:i for i,c in enumerate(all_combinations)}

# Create another dictionary for reverse mapping
itoc={i:c for c, i in ctoi.items()}

# Check if its correct
len(ctoi), ctoi['..'], itoc[1]

(729, 0, '.a')

## Create train, dev and test split

In [6]:
g1 = torch.Generator().manual_seed(42)
perm = torch.randperm(len(words), generator=g1)
perm, perm.shape

(tensor([ 4348, 12372,  7029,  ...,  7956,  2399,  8375]), torch.Size([32033]))

In [7]:
words_shuffled = [words[i] for i in perm]
train_words = words_shuffled[:25626]
dev_words   = words_shuffled[25626:28830]
test_words  = words_shuffled[28830:32033]

## Trigram

In [8]:
# Creating the dataset
def create_dataset_trigram(words):
    xs, ys = [], []
    for w in words:
        chs = ['.','.'] + list(w) + ['.']  # eg ['.', '.', 'e', 'm', 'm', 'a', '.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ch12 = ch1+ch2
            idx12 = ctoi[ch12]
            idx3 = stoi[ch3]
            xs.append(idx12)
            ys.append(idx3)
    
    # Converting dataset into tensor
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    num = xs.numel()
    return xs, ys, num 

In [9]:
xs_train_t, ys_train_t, training_size_t = create_dataset_trigram(train_words)
xs_dev_t, ys_dev_t, dev_size_t = create_dataset_trigram(dev_words)
xs_test_t, ys_test_t, test_size_t = create_dataset_trigram(test_words)

In [10]:
print(xs_dev_t.shape)
print(ys_dev_t.shape)
print(dev_size_t)

torch.Size([22768])
torch.Size([22768])
22768


In [11]:
# CUDA 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Move data and weights to device
xs_train_t = xs_train_t.to(device)
ys_train_t = ys_train_t.to(device)
xs_dev_t = xs_dev_t.to(device)
ys_dev_t = ys_dev_t.to(device)
xs_test_t = xs_test_t.to(device)
ys_test_t = ys_test_t.to(device)


Using: cuda


In [12]:
g = torch.Generator(device=device).manual_seed(312312)
W_trigram = torch.randn((729, 27), generator=g, device=device, requires_grad=True)
W_trigram = W_trigram.to(device)

# E04: Directly Indexing instead of one_hot
we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

In [13]:
lambdaa = 0.5

for i in range(100):

    # Forward pass
    logits = W_trigram[xs_train_t] #(729, 27)[N] → (N, 27)

    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = (-probs[torch.arange(training_size_t, device=device),ys_train_t].log().mean()
        + lambdaa / (2 * training_size_t) * (W_trigram ** 2).sum()
    )

    print(f"Train loss: {loss.item()}")

    # Backward
    W_trigram.grad = None
    loss.backward()

    # Update
    with torch.no_grad():
        W_trigram += -70 * W_trigram.grad

Train loss: 3.7490501403808594
Train loss: 3.6105802059173584
Train loss: 3.52333402633667
Train loss: 3.4512369632720947
Train loss: 3.3870716094970703
Train loss: 3.329087972640991
Train loss: 3.2765185832977295
Train loss: 3.228853702545166
Train loss: 3.185647964477539
Train loss: 3.146470308303833
Train loss: 3.110886573791504
Train loss: 3.0784668922424316
Train loss: 3.048804998397827
Train loss: 3.0215370655059814
Train loss: 2.9963529109954834
Train loss: 2.972993850708008
Train loss: 2.9512457847595215
Train loss: 2.930931329727173
Train loss: 2.911900758743286
Train loss: 2.8940269947052
Train loss: 2.8772008419036865
Train loss: 2.861326217651367
Train loss: 2.8463199138641357
Train loss: 2.832106113433838
Train loss: 2.818618059158325
Train loss: 2.8057966232299805
Train loss: 2.7935874462127686
Train loss: 2.7819416522979736
Train loss: 2.7708168029785156
Train loss: 2.7601735591888428
Train loss: 2.749976396560669
Train loss: 2.740194320678711
Train loss: 2.7307991981506

In [14]:
# Generating names
for i in range(30):
    out = [] # for storing the output
    context = ['.', '.'] # for triggering the model to start outputting characters to form a name
    
    while True:
        pair = context[0] + context[1]
        ix = ctoi[pair]
    
        logits = W_trigram[ix].unsqueeze(0)
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        # Randomly draw one index from the 27 indices, using the values in p as the sampling probabilities.
        # idx with highest probability has higher chances, but its not guaranteed
        next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    
        next_char = itos[next_ix]

        if next_char == '.': 
            break         # breaking when it hits end character
        out.append(next_char)
        context = [context[1], next_char]
    
    if len(out)>1:
        print(''.join(out)) 

tauous
launi
andaleifdwbkno
sequetwci
amir
maylemdulcjmnsonwqwmlelloe
los
malynelumbtmgfvpe
cabelynna
lhwth
la
maloycprgghlmvvydcob
kaodgjrpaejkwavanyianna
chanxo
kail
backsgvncmjszjbarrynee
rea
breslettwnjvvvtuz
afcore
ariah
alain
zayanna
fvzkjdvgu
minna
fzmpkbbaston
ellenijlndrinspnwwnopoluna
taliupkbzizen
wkqsdine
ee


## Evaluating Trigram

In [15]:
# Evaluating on test_set 
logits = W_trigram[xs_test_t]
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_test_t)), ys_test_t]

avg_nll_trigram_test = nlls.mean().item()


print(f'Test loss = {avg_nll_trigram_test}')


Test loss = 2.439318895339966


## Summary
Directly indexing into rows of W does not break the pipeline or add unnecessary behind the scene bug in this case. We just need to unsqueeze the output of Weights[idx] to get logits during inference. The training happens normally as it did with one_hot encoding.
The test loss is 2.4388301372528076.

# E05: Using cross_entropy loss 
look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

Less code so less chances of bug from our side eg shape mismatches. More stable implementation

In [16]:
g2 = torch.Generator(device=device).manual_seed(312312)
W_trigram = torch.randn((729, 27), generator=g2, device=device, requires_grad=True)
W_trigram = W_trigram.to(device)

In [17]:
lambdaa = 0.5

for i in range(100):

    # Forward pass
    logits = W_trigram[xs_train_t] #(729, 27)[N] → (N, 27)
    
    loss = F.cross_entropy(logits, ys_train_t) + lambdaa / (2 * training_size_t) * (W_trigram ** 2).sum()

    print(f"Train loss: {loss.item()}")

    # Backward
    W_trigram.grad = None
    loss.backward()

    # Update
    with torch.no_grad():
        W_trigram += -70 * W_trigram.grad

Train loss: 3.749051332473755
Train loss: 3.6105806827545166
Train loss: 3.523327589035034
Train loss: 3.451239824295044
Train loss: 3.3870749473571777
Train loss: 3.329092502593994
Train loss: 3.2765092849731445
Train loss: 3.228853225708008
Train loss: 3.185647487640381
Train loss: 3.1464650630950928
Train loss: 3.110880136489868
Train loss: 3.0784666538238525
Train loss: 3.0487937927246094
Train loss: 3.021535634994507
Train loss: 2.9963483810424805
Train loss: 2.9729959964752197
Train loss: 2.9512412548065186
Train loss: 2.930933713912964
Train loss: 2.9119036197662354
Train loss: 2.894031047821045
Train loss: 2.8771965503692627
Train loss: 2.861323356628418
Train loss: 2.8463141918182373
Train loss: 2.8321094512939453
Train loss: 2.8186216354370117
Train loss: 2.8058009147644043
Train loss: 2.793588876724243
Train loss: 2.7819440364837646
Train loss: 2.7708253860473633
Train loss: 2.76016902923584
Train loss: 2.7499773502349854
Train loss: 2.740196943283081
Train loss: 2.730801343

In [22]:
# Generating names
for i in range(30):
    out = [] # for storing the output
    context = ['.', '.'] # for triggering the model to start outputting characters to form a name
    
    while True:
        pair = context[0] + context[1]
        ix = ctoi[pair]
    
        logits = W_trigram[ix].unsqueeze(0)
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        # Randomly draw one index from the 27 indices, using the values in p as the sampling probabilities.
        # idx with highest probability has higher chances, but its not guaranteed
        next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g2).item()
    
        next_char = itos[next_ix]

        if next_char == '.': 
            break         # breaking when it hits end character
        out.append(next_char)
        context = [context[1], next_char]
    
    if len(out)>1:
        print(''.join(out)) 

jeqdcjytaqvlee
ck
lee
heddwbmhfkimnzzxtler
remilamari
jer
ketcihintxostale
marlehgwx
nlene
brini
ory
eliaedtqfmyvnoreighfwriwjos
san
jah
ruwnzaexdpmwyhjyvwoxiqkjfwgzdfuielishmlkevuhpcbjn
yejomre
aisxjcvcbrayleenfpfprya
peyuri
anniyah
dqcqiptycdzpen
cacwlpvspjvbudggpe
ambepsdcoohqvpxubksiqtjgb
seden
zari
bxsjgiuaqhaw
alianuxubzjjor
mai
bagypndpi
tathosar
jaiyah


In [23]:
# Evaluating on test_set 
logits = W_trigram[xs_test_t]
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_test_t)), ys_test_t]

avg_nll_trigram_test = nlls.mean().item()


print(f'Test loss = {avg_nll_trigram_test}')


Test loss = 2.439318895339966
